<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_54_typescript_for_ai_frontends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🌐 **Module 54 — TypeScript for AI Frontends (closing the loop — the chat UI you ship to users)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> 🎉 **Final module.** This closes the 54-module arc: from `import pandas` (M1) to a working AI product running in production.


# Module 54 — TypeScript for AI Frontends

> The whole course built the **backend**. M44–M53 stood up vLLM, gRPC, Kubernetes, observability, the polyglot stack. But **users only see the frontend** — the chat box, the streaming tokens, the citations, the tool-call cards. And the language of frontends in 2026 is **TypeScript**.
>
> This module is the missing piece. TypeScript fundamentals, then the **Vercel AI SDK** + **React** combo most production AI apps use, then **streaming**, **tool calls**, **structured outputs**, and **deployment** — wired against the OpenAI-compatible `/v1` endpoint your backend already exposes.

### What you'll cover
1. Why TypeScript — and why every AI startup ships it
2. TS in 5 minutes: types, interfaces, generics, narrowing
3. **The AI SDK landscape** — Vercel AI SDK, OpenAI/Anthropic SDKs, LangChain.js
4. A minimal Node script — call your vLLM backend in TS
5. **Streaming** — Server-Sent Events done right
6. **React + `useChat`** — a real chat UI in 30 lines
7. **Tool calling** + JSON schema (Zod) for typed handoffs
8. **Structured outputs** — `streamObject` for live JSON
9. **Edge runtimes** — Vercel, Cloudflare Workers, Bun
10. Production checklist: auth, rate-limit, abuse, telemetry, cost


## 1 · Why TypeScript

| Reason | Why it matters for AI apps |
|---|---|
| **Static types** on top of JS | typo in `messages[0].rol` is a compile error, not a runtime crash |
| **One language end-to-end** | the same Zod schema validates a tool call on browser, edge, and Node |
| **Massive ecosystem** | every model SDK ships TS first (OpenAI, Anthropic, Mistral, Cohere, Vertex) |
| **Edge runtimes** | TS bundles run on Cloudflare / Vercel / Bun edge — sub-50 ms cold start near the user |
| **Streaming primitives** | `ReadableStream` + `EventSource` + the Web Streams API are first-class |

The reality of 2026: **OpenAI's official Python SDK is excellent, but the TS one is the reference**. Streaming, tool-calling helpers, and the SDK ecosystem land in TS first.


## 2 · TypeScript in 5 minutes

In [ ]:
ts_tour = '''// types: like Python type hints, but enforced
const model: string = "gpt-4o-mini";
const temperature: number = 0.7;
const stream: boolean = true;

// INTERFACE — describe the shape of an object
interface ChatMessage {
  role: "system" | "user" | "assistant" | "tool";   // union of literals
  content: string;
  name?: string;          // optional
}

const msgs: ChatMessage[] = [
  { role: "user", content: "What is RAG?" },
];

// GENERICS — types parameterised by other types
function head<T>(xs: T[]): T | undefined {
  return xs[0];
}
const first = head(msgs);   // inferred as ChatMessage | undefined

// NARROWING — TS understands `if`/`typeof`/`in` checks
function describe(x: string | number) {
  if (typeof x === "string") return x.toUpperCase();   // here TS knows x is string
  return x.toFixed(2);                                  // here it's number
}

// TYPE ALIAS — sometimes nicer than interface
type Tool = { name: string; description: string; schema: unknown };

console.log(first, describe("hi"), describe(3.14159));
'''
print(ts_tour)

**Things that bite Python devs.**
- TS types are **erased** at runtime — no `isinstance(x, ChatMessage)`. For runtime checks use **Zod**.
- `interface` ≈ `type` for object shapes; pick whichever your team prefers.
- `unknown` (safer) vs `any` (the escape hatch — avoid).
- `?:` makes a field optional; `!:` asserts "trust me, it's there".
- `as const` freezes literals, often necessary to keep unions narrow.


## 3 · The AI SDK landscape

| SDK | Best for | Backend support |
|---|---|---|
| **Vercel AI SDK** (`ai`) | full-stack apps with streaming + React/Svelte/Vue hooks | OpenAI, Anthropic, Google, Mistral, Cohere, **OpenAI-compatible (vLLM, Ollama, llama-server)** |
| **OpenAI SDK** (`openai`) | direct OpenAI calls or any compat backend | OpenAI (and via `baseURL` override: vLLM, Ollama, …) |
| **Anthropic SDK** (`@anthropic-ai/sdk`) | Claude tool-use, computer-use | Anthropic |
| **LangChain.js** | port of LangChain (M32) | many — same abstractions as Python |
| **Mastra / Inngest / Trigger.dev** | durable agentic workflows | many |

**Default pick in 2026: the Vercel AI SDK.** It abstracts the streaming protocol so the same TS code works against OpenAI, Anthropic, your local vLLM, or your fine-tuned Ollama model. Swap the `provider` line, ship.

```ts
import { openai } from "@ai-sdk/openai";
import { anthropic } from "@ai-sdk/anthropic";

// against your local vLLM server (M44) — provider points at the OpenAI-compatible endpoint
import { createOpenAI } from "@ai-sdk/openai";
const myVllm = createOpenAI({ baseURL: "http://localhost:8000/v1", apiKey: "x" });
```


## 4 · Minimal Node script — call your vLLM backend

In [ ]:
tsnode = '''// chat.ts — run with `npx tsx chat.ts`
import { generateText } from "ai";
import { createOpenAI } from "@ai-sdk/openai";

const provider = createOpenAI({
  baseURL: process.env.OPENAI_BASE_URL ?? "http://localhost:8000/v1",
  apiKey:  process.env.OPENAI_API_KEY  ?? "x",          // any non-empty string
});

const { text, usage } = await generateText({
  model: provider("Qwen/Qwen2.5-0.5B-Instruct"),
  system: "You are a terse assistant.",
  prompt: "What is PagedAttention?",
});

console.log(text);
console.log("tokens:", usage);
'''
print(tsnode)

**That's the whole script.** `generateText` works against **OpenAI, Anthropic, vLLM (M44), Ollama (M38), llama-server (M38)** with no code change — just swap the provider. The same shape works in Node, Bun, Vercel Edge, Cloudflare Workers.

## 5 · Streaming — Server-Sent Events done right

LLMs feel slow if you wait for the whole response. **Stream tokens** the moment they're produced.

In [ ]:
ts_stream = '''// In a Next.js / React Router / Remix route handler:
import { streamText } from "ai";

export async function POST(req: Request) {
  const { messages } = await req.json();
  const result = await streamText({
    model: provider("Qwen/Qwen2.5-0.5B-Instruct"),
    system: "Answer in one paragraph.",
    messages,
  });
  return result.toDataStreamResponse();   // SSE-compatible Response
}
'''
print(ts_stream)

On the wire that's a stream of tiny JSON-encoded events:
```
data: { "type": "text-delta", "textDelta": "Paged" }
data: { "type": "text-delta", "textDelta": "Attention" }
data: { "type": "text-delta", "textDelta": " is …" }
data: { "type": "finish", "usage": { … } }
```

Browsers consume it natively via **`EventSource`** or `fetch` + `ReadableStream`. The Vercel SDK's React hook (`useChat`) hides the details so you write zero plumbing.

## 6 · React + `useChat` — a chat UI in 30 lines

In [ ]:
ts_react = '''// app/page.tsx — Next.js App Router
"use client";
import { useChat } from "ai/react";

export default function Page() {
  const { messages, input, handleInputChange, handleSubmit, isLoading } = useChat();

  return (
    <main className="mx-auto max-w-2xl p-6">
      <ul className="space-y-2">
        {messages.map(m => (
          <li key={m.id} className={m.role === "user" ? "text-right" : ""}>
            <span className="rounded-2xl px-3 py-2 inline-block bg-gray-100">
              <strong>{m.role}:</strong> {m.content}
            </span>
          </li>
        ))}
      </ul>

      <form onSubmit={handleSubmit} className="flex gap-2 mt-4">
        <input
          value={input}
          onChange={handleInputChange}
          placeholder="Ask anything…"
          className="flex-1 border rounded px-3 py-2"
        />
        <button disabled={isLoading} className="px-4 py-2 rounded bg-black text-white">
          {isLoading ? "…" : "Send"}
        </button>
      </form>
    </main>
  );
}
'''
print(ts_react)

That's a **streaming chat UI**. `useChat()`:
- POSTs to `/api/chat` (the route from §5) automatically.
- Keeps `messages` in React state, **including the streaming partial assistant message**.
- Exposes `isLoading`, `stop()`, `reload()`, `append()` for tool integrations.
- Works with multi-modal attachments out of the box.

In **Svelte**, `useChat` is `useChat()` from `ai/svelte`. In **Vue**, `ai/vue`. Same API.

## 7 · Tool calling + Zod schemas

In [ ]:
ts_tools = '''// app/api/chat/route.ts
import { streamText, tool } from "ai";
import { z } from "zod";

const tools = {
  weather: tool({
    description: "Get current weather for a city",
    parameters: z.object({
      city: z.string().describe("City name, e.g. Paris"),
      units: z.enum(["c","f"]).default("c"),
    }),
    execute: async ({ city, units }) => {
      // … real fetch to a weather API …
      return { city, units, tempC: 21, condition: "sunny" };
    },
  }),
  ragSearch: tool({
    description: "Search the company knowledge base",
    parameters: z.object({ query: z.string(), k: z.number().int().default(5) }),
    execute: async ({ query, k }) => {
      // … call your retriever (M30) over HTTP/gRPC (M45) …
      return { hits: [/* ... */] };
    },
  }),
};

export async function POST(req: Request) {
  const { messages } = await req.json();
  const result = await streamText({
    model: provider("gpt-4o-mini"),
    messages, tools, maxSteps: 5,        // multi-step tool use
  });
  return result.toDataStreamResponse();
}
'''
print(ts_tools)

**Zod is the killer.** One schema is:
- the **runtime validator** for the model's tool call,
- the **TS type** of the `execute` arguments,
- the **JSON schema** sent to the LLM,
- the **prompt description** (`.describe(...)`).

This eliminates the entire "the model invented a field" failure class — bad calls are rejected before your `execute` runs.

## 8 · Structured outputs — `streamObject`

Often you don't want chat — you want **a typed JSON object**. The SDK streams it as it's produced so the UI can show partial state.

In [ ]:
ts_obj = '''import { streamObject } from "ai";
import { z } from "zod";

const Schema = z.object({
  title: z.string(),
  bullets: z.array(z.string()).min(3).max(5),
  confidence: z.number().min(0).max(1),
});

const { partialObjectStream } = await streamObject({
  model: provider("gpt-4o-mini"),
  schema: Schema,
  prompt: "Summarise the latest paper on PagedAttention.",
});

for await (const partial of partialObjectStream) {
  console.log(partial);   // shows the object filling in field by field
}
'''
print(ts_obj)

Internally this uses the model's grammar-constrained JSON mode (M38). Combined with `useObject()` on the React side, you can render forms / cards that fill themselves in live as the model streams.

## 9 · Edge runtimes

| Runtime | What's special |
|---|---|
| **Vercel Edge** | global region, V8 isolates, ~5 ms cold start, full Vercel AI SDK support |
| **Cloudflare Workers** | similar; bundles to ~1 MB; **AI Gateway** + **Workers AI** native |
| **Bun** | server-side, faster than Node, native `Bun.serve` HTTP, drop-in Node compat |
| **Deno / Deno Deploy** | TS-first runtime; clean stdlib; Deno KV |
| **Node 22** | the safe default; widest ecosystem |

**The whole AI SDK is edge-compatible** (no Node-only APIs). One codebase, deploy anywhere — closer to the user means lower TTFT, especially for short prompts where compute isn't the bottleneck.

## 10 · Production checklist

| Concern | Implementation |
|---|---|
| **Auth** | Auth.js / Clerk / Lucia for sessions; check user before every LLM call |
| **Rate limiting** | Upstash Redis (sliding window — same recipe as M37) keyed by user id |
| **Abuse / prompt injection** | strip user-controlled HTML, cap message length, watchlist patterns |
| **Telemetry** | OTel SDK for browsers + edge → Collector (M51) → Langfuse / Datadog |
| **Cost** | log `usage.totalTokens × $/Mtok` per request; budget alerts |
| **Token leakage** | never bundle API keys client-side; do all model calls in server routes |
| **Retries / fallback** | wrap `streamText` with a fallback provider list (vLLM → OpenAI) |
| **Caching** | identical-prompt cache via Upstash (M37) — biggest cost win |
| **Evals** | Promptfoo / Phoenix / Langfuse evaluators on PR against a fixed dataset |
| **Accessibility** | screen-reader-friendly streaming (announce "assistant typing"); keyboard support |

### Anti-patterns
- ❌ Calling the LLM directly from the browser with the API key. Always proxy through your server.
- ❌ Building chat UI from scratch for the 100th time. Use `useChat` (or shadcn-chat / chat-ui templates).
- ❌ Skipping Zod, then debugging "model returned a field that doesn't exist."
- ❌ Streaming as one big chunk on the server then writing it all out at once. Keep the pipe open end-to-end.
- ❌ Hard-coding `gpt-4o`. Use a config-driven provider so you can swap to your fine-tuned vLLM model (M39 + M44) without redeploying.


## 🎓 The whole arc — what you've built across 54 modules

```
M1-M5     Python · Pandas · NumPy · Plotting · SQL fundamentals
M6-M15    Stats · classical ML · evaluation · feature engineering · MLOps basics
M16-M24   PyTorch · transformers from scratch · diffusion · time-series · math + DeepSeek-V3 internals
M25-M27   Pydantic · LangChain · Chroma RAG starter
M28-M31   FastAPI · serving · vector DBs · advanced RAG
M32-M35   LangChain agents · LangGraph · LlamaIndex · DSPy
M36-M40   PostgreSQL+pgvector · Redis · Ollama · Unsloth fine-tune · TF/Keras
M41-M45   CUDA · vector DB comparison · multi-agent · vLLM · gRPC
M46-M51   Kubernetes · Helm · Terraform · Ansible · Prometheus · OpenTelemetry
M52-M54   Go backends · Rust hot paths · TypeScript frontends
```

You can:
- Train, fine-tune, and quantise a model.
- Serve it at scale with PagedAttention + continuous batching.
- Wrap it in a multi-agent orchestrator with tool calling.
- Deploy the platform on Kubernetes with full Day-2 ops.
- Observe every request end-to-end with metrics + traces.
- Ship a streaming chat UI to users.

That's the whole stack of an AI company in 2026.

## ✅ Recap (M54)
- TypeScript = static types + huge SDK ecosystem + edge-runtime support.
- The **Vercel AI SDK** is the modern default; provider-swappable across OpenAI / Anthropic / vLLM / Ollama.
- Streaming is one line on the server (`streamText().toDataStreamResponse()`) and one hook on the client (`useChat`).
- **Zod schemas** double as runtime validators, TS types, and JSON-schema for tool calls.
- Always proxy LLM calls through your server. Auth, rate-limit, telemetry, budget — non-optional.

## 🙏 What now?
- ⭐ Star the repo: [`github.com/kader-xai/data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)
- 🛠️ Pick **one** module and **build something real** with it. Shipping is the only feedback loop that matters.
- 🌐 Companion site: [`kader-xai.github.io/data-science-roadmap`](https://kader-xai.github.io/data-science-roadmap/)

You're done. Go build.
